# EXACT 2026 Kaggle Inline End-to-End Notebook

Notebook này gom pipeline Type 1 logic vào một file duy nhất để chạy trên Kaggle. Mục tiêu là không phụ thuộc vào package `exact` trong repo: schema, parser, LLM autoformalizer, KB, symbolic solver, router và batch runner đều được inline trong các cell bên dưới.

Luồng chính:
1. Đọc inference JSON từ `/kaggle/input/**/Logic_Based_Educational_Queries_inference.json` hoặc fallback về file local trong repo.
2. Route từng instance sang Type 1 nếu có `premises-NL`, còn Type 2 trả placeholder ổn định.
3. Với Type 1, ưu tiên LLM local/OpenAI-compatible để dịch NL sang Horn IR.
4. Nếu LLM không có hoặc output lỗi và `REQUIRE_LLM=False`, fallback về heuristic parser.
5. Chạy forward chaining solver, xuất response nội bộ và official submission fields.

## 1. Runtime Configuration

Chỉnh các biến dưới đây trước khi chạy toàn bộ notebook trên Kaggle. Nếu bạn có local OpenAI-compatible server, set environment variables trong Kaggle hoặc sửa trực tiếp:

- `EXACT_LLM_PROVIDER`: `none`, `local`, hoặc `openai`. `local` load trực tiếp bằng Transformers.
- `EXACT_LLM_MODEL`: Hugging Face model id hoặc đường dẫn model trong `/kaggle/input/...`
- `EXACT_LLM_BASE_URL`: chỉ cần khi `EXACT_LLM_PROVIDER=openai`, ví dụ `http://localhost:8000/v1`
- `EXACT_LLM_API_KEY`: optional, mặc định là `EMPTY`
- `EXACT_LIMIT`: optional, dùng để smoke test nhanh trước khi chạy full

In [16]:
import dataclasses
import enum
import glob
import hashlib
import importlib.util
import json
import os
import pathlib
import re
import time
import unicodedata
import urllib.error
import urllib.request

INPUT_PATH = (
    "../datasets/exact/Logic_Based_Educational_Queries_inference.json"
)
OUTPUT_PATH = "../../../outputs/logic/predictions.json"
LOCAL_FALLBACK_INPUT = INPUT_PATH

# Hardcoded Kaggle mode: load the model directly with Transformers.
USE_LLM = True
REQUIRE_LLM = True
LLM_PROVIDER = "local"
LLM_BASE_URL = None
LLM_API_KEY = "EMPTY"
LLM_TIMEOUT_SECONDS = 60.0
LLM_MAX_TOKENS = 2048
LLM_TEMPERATURE = 0.0
PROGRESS_EVERY = 100
LIMIT = None

MODEL_CANDIDATES = (
    "Qwen/Qwen2.5-1.5B-Instruct",
)


def resolve_local_model(candidates: tuple[str, ...]) -> str:
    """Return the first Kaggle model folder, or the Hugging Face model id."""

    for candidate in candidates:
        if candidate.startswith("/kaggle/input") and pathlib.Path(candidate).exists():
            return candidate
    return candidates[-1]


LLM_MODEL = resolve_local_model(MODEL_CANDIDATES)

print(
    "Config:",
    {
        "use_llm": USE_LLM,
        "require_llm": REQUIRE_LLM,
        "llm_provider": LLM_PROVIDER,
        "llm_base_url": LLM_BASE_URL,
        "llm_model": LLM_MODEL,
        "limit": LIMIT,
        "output": OUTPUT_PATH,
    },
)

Config: {'use_llm': True, 'require_llm': True, 'llm_provider': 'local', 'llm_base_url': None, 'llm_model': 'Qwen/Qwen2.5-1.5B-Instruct', 'limit': None, 'output': '../../../outputs/logic/predictions.json'}


## 2. Shared Schemas

Các schema này thay phần `exact.common.schemas`. Dùng `dataclass` để notebook tự chạy được trên Kaggle mà không cần Pydantic hoặc package nội bộ.

In [17]:
class TaskType(str, enum.Enum):
    """Top-level task family selected by the router."""

    TYPE1_LOGIC = "type1_logic"
    TYPE2_PHYSICS = "type2_physics"
    UNKNOWN = "unknown"


class QuestionType(str, enum.Enum):
    """Question shape used by task-specific pipelines."""

    MCQ = "mcq"
    YES_NO_UNCERTAIN = "yes_no_uncertain"
    OPEN_ENDED = "open_ended"
    NUMERICAL = "numerical"
    UNKNOWN = "unknown"


@dataclasses.dataclass(frozen=True)
class PredictionRequest:
    """Normalized input sample for either Type 1 logic or Type 2 physics."""

    question: str
    id: str | None = None
    premises_nl: list[str] | None = None
    raw: dict | None = None

    @classmethod
    def from_dict(cls, payload: dict) -> "PredictionRequest":
        """Build a request from organizer JSON with lenient extra fields."""

        question = str(payload.get("question") or "").strip()
        if not question:
            raise ValueError("question must not be empty")
        premises = payload.get("premises-NL") or payload.get("premises_nl")
        if premises is not None:
            premises = [str(item).strip() for item in premises if str(item).strip()]
        return cls(
            id=payload.get("id"),
            question=question,
            premises_nl=premises,
            raw=payload,
        )

    @property
    def inferred_task_type(self) -> TaskType:
        """Infer task type from the presence of natural-language premises."""

        if self.premises_nl:
            return TaskType.TYPE1_LOGIC
        return TaskType.TYPE2_PHYSICS


@dataclasses.dataclass(frozen=True)
class PredictionResponse:
    """Competition-facing prediction plus local debugging metadata."""

    answer: str
    explanation: str
    fol: str | None = None
    cot: list[str] | None = None
    premises: list[str] | None = None
    confidence: float | None = None
    id: str | None = None
    task_type: TaskType | None = None
    question_type: QuestionType = QuestionType.UNKNOWN
    unit: str | None = None
    error: str | None = None

    def to_dict(self) -> dict:
        """Return a JSON-serializable internal response."""

        data = dataclasses.asdict(self)
        if isinstance(self.task_type, TaskType):
            data["task_type"] = self.task_type.value
        if isinstance(self.question_type, QuestionType):
            data["question_type"] = self.question_type.value
        return data


def to_official_response(response: PredictionResponse) -> dict:
    """Convert an internal response into the stable EXACT submission shape."""

    return {
        "answer": response.answer,
        "explanation": response.explanation,
        "fol": response.fol,
        "cot": response.cot,
        "premises": response.premises,
        "confidence": response.confidence,
    }

## 3. Logic IR

Đây là IR nhỏ kiểu Horn clause. LLM hoặc heuristic parser đều dịch premise/question về các object này, sau đó solver deterministic xử lý proof.

In [18]:
@dataclasses.dataclass(frozen=True, order=True)
class Atom:
    """A normalized propositional or predicate-like logical statement."""

    pred: str
    args: tuple[str, ...] = ()
    negated: bool = False
    text: str | None = dataclasses.field(default=None, compare=False)

    def positive(self) -> "Atom":
        """Return the non-negated version of this atom."""

        return Atom(pred=self.pred, args=self.args, negated=False, text=self.text)

    def negation(self) -> "Atom":
        """Return the logical negation of this atom."""

        return Atom(
            pred=self.pred,
            args=self.args,
            negated=not self.negated,
            text=self.text,
        )

    def display(self) -> str:
        """Render the atom for explanations and FOL-like debugging text."""

        label = self.text or (
            f"{self.pred}({', '.join(self.args)})"
            if self.args
            else self.pred.replace("_", " ")
        )
        return f"not {label}" if self.negated else label


@dataclasses.dataclass(frozen=True)
class Rule:
    """A Horn-style rule: all conditions must hold to derive conclusion."""

    conditions: tuple[Atom, ...]
    conclusion: Atom
    source_idx: int
    text: str


@dataclasses.dataclass(frozen=True)
class Fact:
    """A directly stated premise fact."""

    atom: Atom
    source_idx: int
    text: str


@dataclasses.dataclass(frozen=True)
class ProofStep:
    """One derivation step with provenance for explanation and scoring depth."""

    derived: Atom
    used_premises: tuple[int, ...]
    rule_idx: int | None
    parents: tuple[Atom, ...] = ()
    natural_language: str | None = None


@dataclasses.dataclass(frozen=True)
class ParsedPremise:
    """Parser output for one source premise."""

    facts: tuple[Fact, ...] = ()
    rules: tuple[Rule, ...] = ()
    warnings: tuple[str, ...] = ()


@dataclasses.dataclass(frozen=True)
class Query:
    """Normalized target claim for yes/no/unknown reasoning."""

    claim: Atom
    raw_question: str
    expects_negation: bool = False


@dataclasses.dataclass(frozen=True)
class SolveResult:
    """Result returned by a symbolic solver."""

    label: str
    claim: Atom
    proof: tuple[ProofStep, ...] = ()
    supporting_premises: tuple[int, ...] = ()
    mode: str = "symbolic_forward_chain"
    warnings: tuple[str, ...] = ()


@dataclasses.dataclass(frozen=True)
class Theory:
    """Extension point for richer typed FOL/Z3 encodings."""

    sorts: dict[str, list[str]] = dataclasses.field(default_factory=dict)
    predicates: dict[str, tuple[str, ...]] = dataclasses.field(default_factory=dict)
    functions: dict[str, tuple[tuple[str, ...], str]] = dataclasses.field(default_factory=dict)
    constants: dict[str, str] = dataclasses.field(default_factory=dict)

## 4. Heuristic Horn Parser

Fallback parser cho các pattern phổ biến trong Type 1. Khi LLM chạy tốt, phần này chủ yếu là safety net; khi local endpoint lỗi, notebook vẫn có thể chạy full batch.

In [19]:
_IF_THEN_RE = re.compile(
    r"^\s*if\s+(.+?)\s*,?\s+then\s+(.+?)\.?\s*$",
    re.IGNORECASE,
)
_IF_COMMA_RE = re.compile(r"^\s*if\s+(.+?)\s*,\s+(.+?)\.?\s*$", re.IGNORECASE)
_RELATIVE_RULE_RE = re.compile(
    r"^\s*(?:all\s+)?(?P<subject>[a-z][a-z\s-]*?)\s+who\s+"
    r"(?P<conditions>.+?)\s+"
    r"(?P<conclusion>are|is|can|qualify|qualifies|receive|receives)\s+"
    r"(?P<tail>.+?)\.?\s*$",
    re.IGNORECASE,
)
_TRAILING_PUNCT_RE = re.compile(r"[\s.?!:;]+$")
_WORD_RE = re.compile(r"[a-z0-9]+")

_ENTITY_PREFIXES = {"professor", "dr", "student", "faculty member"}
_SUBJECT_NOUNS = {
    "student",
    "students",
    "faculty member",
    "faculty members",
    "python code",
    "python project",
    "python projects",
    "python",
    "code",
    "project",
    "projects",
}
_SYNONYMS = {
    "research methodology course": "research methodology",
    "required community service hours": "community service",
    "university scholarship": "scholarship",
    "the university scholarship": "scholarship",
    "international program": "international program",
    "the international program": "international program",
    "science assessment": "science assessment",
    "the science assessment": "science assessment",
    "core curriculum": "core curriculum",
    "the core curriculum": "core curriculum",
    "capstone project": "capstone project",
    "a capstone project": "capstone project",
    "honors diploma": "honors diploma",
    "an honors diploma": "honors diploma",
    "faculty recommendation": "faculty recommendation",
    "a faculty recommendation": "faculty recommendation",
    "phd": "phd",
    "a phd": "phd",
    "pep 8 standards": "pep8",
    "pep 8 standard": "pep8",
    "clean and readable code": "clean_readable_code",
    "clean readable code": "clean_readable_code",
}
_LEADING_VERBS = {
    "are",
    "is",
    "was",
    "were",
    "be",
    "been",
    "can",
    "could",
    "do",
    "does",
    "did",
    "has",
    "have",
    "had",
    "completed",
    "passed",
    "received",
    "awarded",
    "qualified",
    "qualify",
    "qualifies",
    "eligible",
    "holds",
    "hold",
    "maintains",
    "maintain",
    "follows",
    "follow",
    "completes",
    "complete",
    "graduates",
    "graduate",
}
_FILLER_WORDS = {
    "a",
    "an",
    "the",
    "all",
    "any",
    "required",
    "academic",
    "standards",
    "standard",
    "hours",
    "her",
    "his",
    "their",
}


def parse_premise_to_ir(premise: str, source_idx: int) -> ParsedPremise:
    """Parse one natural-language premise into facts or Horn rules."""

    text = premise.strip()
    if not text:
        return ParsedPremise(warnings=(f"premise {source_idx + 1} is empty",))

    rule = _parse_rule(text, source_idx)
    if rule is not None:
        return ParsedPremise(rules=(rule,))

    return ParsedPremise(
        facts=(Fact(atom=_atom_from_clause(text), source_idx=source_idx, text=text),)
    )


def parse_question_to_query(question: str) -> Query:
    """Parse a yes/no-style question into a target claim atom."""

    raw = question.strip()
    normalized = _strip_question_shell(raw)
    atom = _atom_from_clause(normalized)
    return Query(claim=atom, raw_question=raw, expects_negation=atom.negated)


def atom_from_text(text: str) -> Atom:
    """Convert a short claim or MCQ option into the canonical Atom form."""

    return _atom_from_clause(text)


def _parse_rule(text: str, source_idx: int) -> Rule | None:
    """Return a Horn rule for conditional and relative-clause patterns."""

    if_then = _IF_THEN_RE.match(text)
    if if_then:
        antecedent, consequent = if_then.groups()
        return _rule_from_parts(antecedent, consequent, source_idx, text)

    if_comma = _IF_COMMA_RE.match(text)
    if if_comma and " then " not in text.lower():
        antecedent, consequent = if_comma.groups()
        return _rule_from_parts(antecedent, consequent, source_idx, text)

    relative = _RELATIVE_RULE_RE.match(_clean_clause(text))
    if relative:
        conditions = tuple(
            _atom_from_clause(part, default_arg="?x")
            for part in _split_conditions(relative.group("conditions"))
        )
        conclusion_text = f"{relative.group('conclusion')} {relative.group('tail')}"
        conclusion = _atom_from_clause(conclusion_text, default_arg="?x")
        return Rule(
            conditions=conditions,
            conclusion=conclusion,
            source_idx=source_idx,
            text=text,
        )

    return None


def _rule_from_parts(
    antecedent: str,
    consequent: str,
    source_idx: int,
    text: str,
) -> Rule:
    """Build a rule from antecedent and consequent text spans."""

    conditions = tuple(
        _atom_from_clause(part, default_arg="?x")
        for part in _split_conditions(antecedent)
    )
    conclusion = _atom_from_clause(consequent, default_arg="?x")
    return Rule(
        conditions=conditions,
        conclusion=conclusion,
        source_idx=source_idx,
        text=text,
    )


def _atom_from_clause(text: str, default_arg: str | None = None) -> Atom:
    """Build an Atom by extracting an optional entity and canonical predicate."""

    clause = _clean_clause(text)
    negated, clause = _strip_negation(clause)
    clause = _strip_subject_intro(clause)

    entity, predicate_clause = _extract_entity(clause)
    arg = entity or default_arg
    pred = _predicate_from_clause(predicate_clause)
    if not pred:
        pred = _slugify(_canonicalize_clause(clause)) or "unknown"

    args = (arg,) if arg else ()
    return Atom(pred=pred, args=args, negated=negated, text=clause)


def _split_conditions(text: str) -> list[str]:
    """Split conjunctions while preserving phrases such as clean and readable."""

    normalized = _clean_clause(text)
    normalized = re.sub(r"\b(?:they|who|it)\b", "", normalized)
    regex = (
        r"\s+(?:and|&)\s+"
        r"(?=(?:has|have|had|passed|completed|received|holds?|maintains?|"
        r"can|is|are|does|do|follows?|qualifies?|awarded)\b)"
    )
    parts = re.split(regex, normalized)
    return [part.strip(" ,") for part in parts if part.strip(" ,")]


def _strip_question_shell(question: str) -> str:
    """Remove question framing so the remaining text is a claim."""

    text = _clean_clause(question)
    text = re.sub(r"^based on the above premises,\s*", "", text)
    text = re.sub(r",?\s*according to the premises$", "", text)
    text = re.sub(r"^does\s+it\s+follow\s+that\s+", "", text)
    text = re.sub(
        r"^(?:does|do|did|is|are|can|could|will|would|should)\s+",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s+(?:hold|holds|follow|follows)$", "", text)
    return text.strip()


def _strip_negation(clause: str) -> tuple[bool, str]:
    """Detect common explicit negation markers and remove them."""

    for prefix in (
        "it is not true that ",
        "not ",
        "does not ",
        "do not ",
        "did not ",
        "cannot ",
    ):
        if clause.startswith(prefix):
            return True, clause[len(prefix) :].strip()

    replacements = (
        (" does not ", " "),
        (" do not ", " "),
        (" did not ", " "),
        (" cannot ", " can "),
        (" is not ", " is "),
        (" are not ", " are "),
    )
    for marker, replacement in replacements:
        if marker in clause:
            return True, clause.replace(marker, replacement, 1).strip()

    return False, clause


def _strip_subject_intro(clause: str) -> str:
    """Remove generic subject introductions that should not become predicates."""

    return re.sub(
        r"^(?:all\s+)?(?:students?|faculty members?|python projects?|"
        r"python code|a student|a faculty member)\s+",
        "",
        clause,
    ).strip()


def _extract_entity(clause: str) -> tuple[str | None, str]:
    """Extract a named entity constant from the start of a clause when present."""

    words = clause.split()
    if not words:
        return None, clause

    if len(words) >= 2 and " ".join(words[:2]) in {"professor john", "professor sophia"}:
        return _slugify(" ".join(words[:2])), " ".join(words[2:])

    first = words[0]
    blocked = _SUBJECT_NOUNS | _LEADING_VERBS | _FILLER_WORDS | {"if", "then", "they", "it"}
    if first not in blocked:
        if first in _ENTITY_PREFIXES and len(words) >= 2:
            return _slugify(" ".join(words[:2])), " ".join(words[2:])
        return _slugify(first), " ".join(words[1:])

    return None, clause


def _predicate_from_clause(clause: str) -> str:
    """Normalize educational verb phrases into stable predicate names."""

    clause = _canonicalize_clause(clause)
    patterns = (
        (r"^(?:has|have|had) completed (.+)$", "completed"),
        (r"^completed (.+)$", "completed"),
        (r"^(?:has|have|had) passed (.+)$", "passed"),
        (r"^passed (.+)$", "passed"),
        (r"^(?:has|have|had) received (.+)$", "received"),
        (r"^received (.+)$", "received"),
        (r"^(?:has|have|had) been awarded (.+)$", "awarded"),
        (r"^(?:is|are|was|were) awarded (.+)$", "awarded"),
        (r"^awarded (.+)$", "awarded"),
        (r"^(?:is|are|was|were) qualified for (.+)$", "qualified_for"),
        (r"^qualified for (.+)$", "qualified_for"),
        (r"^(?:is|are|was|were) eligible for (.+)$", "eligible_for"),
        (r"^eligible for (.+)$", "eligible_for"),
        (r"^qualif(?:y|ies) for (.+)$", "qualifies_for"),
        (r"^(?:can|could) teach (.+)$", "can_teach"),
        (r"^(?:can|could) supervise (.+)$", "can_supervise"),
        (r"^(?:can|could) serve on (.+)$", "can_serve_on"),
        (r"^(?:can|could) propose (.+)$", "can_propose"),
        (r"^holds? (.+)$", "holds"),
        (r"^maintains? (?:a )?gpa (?:above|of) (.+)$", "maintains_gpa"),
        (r"^follows? (.+)$", "follows"),
        (r"^(?:is|are|was|were) (.+)$", ""),
        (r"^(?:must be|needs? to be) (.+)$", ""),
        (r"^needs? (?:to pass|to complete|to receive)?\s*(.+)$", "needs"),
    )

    for regex, prefix in patterns:
        match = re.match(regex, clause)
        if not match:
            continue
        obj = _normalize_object(match.group(1))
        if not obj:
            return prefix
        return f"{prefix}_{obj}" if prefix else obj

    return _normalize_object(clause)


def _normalize_object(text: str) -> str:
    """Normalize noun/adjective objects into slug-safe predicate suffixes."""

    value = _canonicalize_clause(text)
    value = _SYNONYMS.get(value, value)
    tokens = [token for token in _WORD_RE.findall(value) if token not in _FILLER_WORDS]
    return "_".join(tokens)


def _clean_clause(text: str) -> str:
    """Normalize punctuation, Unicode, whitespace, and casing."""

    text = unicodedata.normalize("NFKC", str(text))
    text = text.strip().strip("\"'")
    text = _TRAILING_PUNCT_RE.sub("", text)
    text = text.replace("’", "'")
    return re.sub(r"\s+", " ", text).lower()


def _canonicalize_clause(text: str) -> str:
    """Remove lightweight articles and truth framing from a clause."""

    text = re.sub(r"^(?:a|an|the)\s+", "", text.strip())
    text = re.sub(r"\s+(?:is|are|was|were)\s+true$", "", text)
    return text.strip()


def _slugify(text: str) -> str:
    """Convert text to lowercase underscore tokens."""

    return "_".join(_WORD_RE.findall(str(text).lower()))

## 5. Knowledge Base and Forward-Chaining Solver

Phần này gom logic từ `kb.py` và `symbolic_solvers/forward_chain/solver.py`. Solver chỉ suy diễn Horn clauses nên nhanh, deterministic và dễ trace premise support.

In [20]:
PARSER_VERSION = "inline_llm_first_horn_v1"
_KB_CACHE: dict[str, "KnowledgeBase"] = {}


@dataclasses.dataclass(frozen=True)
class KnowledgeBase:
    """Parsed, reusable premise set with source-preserving facts and rules."""

    raw_premises: tuple[str, ...]
    facts: tuple[Fact, ...]
    rules: tuple[Rule, ...]
    premise_hash: str
    parser_version: str = PARSER_VERSION
    theory: Theory | None = None
    warnings: tuple[str, ...] = ()


def hash_premises(
    premises: list[str] | tuple[str, ...],
    parser_version: str = PARSER_VERSION,
) -> str:
    """Hash premises plus parser version so stale KBs do not leak."""

    text = parser_version + "\n" + "\n".join(premises)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def build_kb_from_premises(
    premises: list[str] | tuple[str, ...],
    premise_hash: str | None = None,
    parser_version: str = PARSER_VERSION,
) -> KnowledgeBase:
    """Build a KB by parsing premises with the heuristic parser."""

    parsed = tuple(parse_premise_to_ir(item, idx) for idx, item in enumerate(premises))
    return build_kb_from_parsed_premises(
        premises=premises,
        parsed_premises=parsed,
        premise_hash=premise_hash,
        parser_version=parser_version,
    )


def build_kb_from_parsed_premises(
    premises: list[str] | tuple[str, ...],
    parsed_premises: tuple[ParsedPremise, ...],
    premise_hash: str | None = None,
    parser_version: str = PARSER_VERSION,
    extra_warnings: tuple[str, ...] = (),
) -> KnowledgeBase:
    """Build a KB from already translated premise IR."""

    facts: list[Fact] = []
    rules: list[Rule] = []
    warnings: list[str] = list(extra_warnings)
    raw_premises = tuple(premises)

    for parsed in parsed_premises:
        facts.extend(parsed.facts)
        rules.extend(parsed.rules)
        warnings.extend(parsed.warnings)

    return KnowledgeBase(
        raw_premises=raw_premises,
        facts=tuple(facts),
        rules=tuple(rules),
        premise_hash=premise_hash or hash_premises(raw_premises, parser_version),
        parser_version=parser_version,
        warnings=tuple(warnings),
    )


def get_or_build_kb(premises: list[str] | tuple[str, ...]) -> KnowledgeBase:
    """Return a cached heuristic KB for repeated premise groups."""

    key = hash_premises(premises)
    if key not in _KB_CACHE:
        _KB_CACHE[key] = build_kb_from_premises(premises, premise_hash=key)
    return _KB_CACHE[key]


def clear_kb_cache() -> None:
    """Clear the in-memory KB cache."""

    _KB_CACHE.clear()


@dataclasses.dataclass(frozen=True)
class ForwardChainSolver:
    """Derive all Horn consequences and answer by proof lookup."""

    name: str = "forward_chain_horn"

    def solve(self, kb: KnowledgeBase, claim: Atom) -> SolveResult:
        """Solve a single claim against a KB."""

        return solve_query(kb, claim, mode=self.name)


def solve_query(kb: KnowledgeBase, claim: Atom, mode: str = "forward_chain_horn") -> SolveResult:
    """Prove claim, prove its negation, or return Unknown."""

    known, proofs = derive_closure(kb)

    if claim in known:
        proof = _trace_proof(claim, proofs)
        return SolveResult(
            label="Yes",
            claim=claim,
            proof=tuple(proof),
            supporting_premises=_support_from_proof(proof),
            mode=mode,
            warnings=kb.warnings,
        )

    negated_claim = claim.negation()
    if negated_claim in known:
        proof = _trace_proof(negated_claim, proofs)
        return SolveResult(
            label="No",
            claim=claim,
            proof=tuple(proof),
            supporting_premises=_support_from_proof(proof),
            mode=mode,
            warnings=kb.warnings,
        )

    return SolveResult(
        label="Unknown",
        claim=claim,
        proof=(),
        supporting_premises=(),
        mode=mode,
        warnings=kb.warnings,
    )


def derive_closure(kb: KnowledgeBase) -> tuple[set[Atom], dict[Atom, ProofStep]]:
    """Derive all reachable atoms and keep the first proof for each atom."""

    known: set[Atom] = set()
    proofs: dict[Atom, ProofStep] = {}

    for fact in kb.facts:
        if fact.atom not in known:
            known.add(fact.atom)
            proofs[fact.atom] = ProofStep(
                derived=fact.atom,
                used_premises=(fact.source_idx,),
                rule_idx=None,
                parents=(),
                natural_language=(
                    f"Premise {fact.source_idx + 1} states {fact.atom.display()}."
                ),
            )

    changed = True
    while changed:
        changed = False
        for rule in kb.rules:
            for binding, parents in _match_conditions(rule.conditions, tuple(known)):
                conclusion = apply_subst(rule.conclusion, binding)
                if not _is_ground(conclusion) or conclusion in known:
                    continue

                known.add(conclusion)
                parent_premises: list[int] = [rule.source_idx]
                for parent in parents:
                    parent_premises.extend(proofs[parent].used_premises)
                proofs[conclusion] = ProofStep(
                    derived=conclusion,
                    used_premises=tuple(sorted(set(parent_premises))),
                    rule_idx=rule.source_idx,
                    parents=parents,
                    natural_language=(
                        f"Premise {rule.source_idx + 1} derives "
                        f"{conclusion.display()} when "
                        f"{', '.join(parent.display() for parent in parents)} hold."
                    ),
                )
                changed = True

    return known, proofs


def is_variable(term: str) -> bool:
    """Return whether a term is a rule variable."""

    return term.startswith("?")


def unify(
    pattern: Atom,
    ground_fact: Atom,
    subst: dict[str, str] | None = None,
) -> dict[str, str] | None:
    """Unify a rule pattern with a ground fact under an optional subst."""

    if (
        pattern.pred != ground_fact.pred
        or pattern.negated != ground_fact.negated
        or len(pattern.args) != len(ground_fact.args)
    ):
        return None

    bindings = dict(subst or {})
    for pattern_term, fact_term in zip(pattern.args, ground_fact.args):
        if is_variable(pattern_term):
            resolved = _resolve_term(pattern_term, bindings)
            if resolved != pattern_term:
                if resolved != fact_term:
                    return None
            else:
                bindings[pattern_term] = fact_term
        elif pattern_term != fact_term:
            return None

    return bindings


def apply_subst(atom: Atom, subst: dict[str, str]) -> Atom:
    """Apply variable bindings to an atom."""

    return Atom(
        pred=atom.pred,
        args=tuple(_resolve_term(arg, subst) for arg in atom.args),
        negated=atom.negated,
        text=atom.text,
    )


def _match_conditions(
    conditions: tuple[Atom, ...],
    known: tuple[Atom, ...],
    subst: dict[str, str] | None = None,
    parents: tuple[Atom, ...] = (),
):
    """Yield substitutions satisfying all conditions against known facts."""

    if not conditions:
        yield dict(subst or {}), parents
        return

    current, *remaining = conditions
    for candidate in known:
        next_subst = unify(current, candidate, subst)
        if next_subst is None:
            continue
        grounded_parent = apply_subst(current, next_subst)
        if grounded_parent != candidate:
            continue
        yield from _match_conditions(
            tuple(remaining),
            known,
            next_subst,
            (*parents, candidate),
        )


def _is_ground(atom: Atom) -> bool:
    """Return whether all atom arguments are constants."""

    return all(not is_variable(arg) for arg in atom.args)


def _resolve_term(term: str, subst: dict[str, str]) -> str:
    """Resolve chained variable bindings safely."""

    seen: set[str] = set()
    while is_variable(term) and term in subst and term not in seen:
        seen.add(term)
        next_term = subst[term]
        if next_term == term:
            break
        term = next_term
    return term


def _trace_proof(target: Atom, proofs: dict[Atom, ProofStep]) -> list[ProofStep]:
    """Return proof steps in parent-before-child order."""

    ordered: list[ProofStep] = []
    visited: set[Atom] = set()

    def visit(atom: Atom) -> None:
        """Visit proof parents recursively."""

        if atom in visited or atom not in proofs:
            return
        visited.add(atom)
        for parent in proofs[atom].parents:
            visit(parent)
        ordered.append(proofs[atom])

    visit(target)
    return ordered


def _support_from_proof(proof: list[ProofStep]) -> tuple[int, ...]:
    """Collect sorted premise indices from proof steps."""

    support: set[int] = set()
    for step in proof:
        support.update(step.used_premises)
    return tuple(sorted(support))

## 6. Explanation Helpers

Sinh `explanation`, `cot`, `premises`, và `fol` theo format organizer yêu cầu. Đây là phần nên giữ ngắn, traceable và không hallucinate ngoài premise.

In [21]:
def explain_result(result: SolveResult, kb: KnowledgeBase) -> tuple[str, list[str], list[str]]:
    """Return explanation, CoT-style trace, and cited premise labels."""

    premise_labels = [f"P{idx + 1}" for idx in result.supporting_premises]

    if result.label == "Unknown":
        return (
            "The provided premises do not prove the claim or its negation, "
            "so the answer is Unknown.",
            ["No symbolic proof was found for the claim or for its negation."],
            [],
        )

    cot = [step.natural_language or f"Derived {step.derived.display()}." for step in result.proof]
    support_text = ", ".join(premise_labels) if premise_labels else "the parsed premises"

    if result.label == "Yes":
        explanation = f"Using {support_text}, the symbolic proof derives {result.claim.display()}."
    else:
        explanation = (
            f"Using {support_text}, the symbolic proof derives the negation of "
            f"{result.claim.display()}."
        )

    return explanation, cot, premise_labels


def kb_to_fol_like_text(kb: KnowledgeBase) -> str:
    """Build readable formalization for the optional `fol` field."""

    lines: list[str] = []
    for fact in kb.facts:
        lines.append(f"P{fact.source_idx + 1}: {fact.atom.display()}")
    for rule in kb.rules:
        conditions = " AND ".join(condition.display() for condition in rule.conditions)
        lines.append(f"P{rule.source_idx + 1}: {conditions} -> {rule.conclusion.display()}")
    return "\n".join(lines)

## 7. LLM Autoformalizer

Phần này thay `llm_client.py` và `logic/llm_translator.py`, nhưng được viết lại theo hướng notebook tự chạy. Nó gọi local/OpenAI-compatible chat completions bằng `urllib` chuẩn của Python, yêu cầu JSON autoformalization có `predicates`, `premises`, `query`, `options`, rồi repair/normalize output trước khi convert sang `Atom`, `Fact`, `Rule`.

Điểm quan trọng cho local model: nếu model trả JSON gần đúng nhưng lệch key nhẹ (`condition`, `then`, `predicate`, query không bọc `claim`, option dùng `claim` thay vì `goal`, thiếu `text`), normalizer sẽ sửa trước khi fallback heuristic.

In [22]:
_PREDICATE_NAME_RE = re.compile(r"^[a-z][a-z0-9_]*$")
_VARIABLE_NAME_RE = re.compile(r"^\?[a-z][a-z0-9_]*$")
_CONSTANT_NAME_RE = re.compile(r"^[a-z][a-z0-9_]*$")
_OPTION_LABELS = {"A", "B", "C", "D"}


@dataclasses.dataclass(frozen=True)
class TranslationResult:
    """Output of NL-to-IR translation, including optional MCQ option atoms."""

    parsed_premises: tuple[ParsedPremise, ...]
    query: Query
    warnings: tuple[str, ...] = ()
    option_atoms: dict[str, Atom] = dataclasses.field(default_factory=dict)


class OpenAICompatibleJsonClient:
    """Small OpenAI-compatible JSON chat client for hosted endpoints."""

    def __init__(
        self,
        base_url: str,
        model: str,
        api_key: str = "EMPTY",
        timeout_seconds: float = 60.0,
    ) -> None:
        """Store endpoint configuration for synchronous JSON calls."""

        self.base_url = base_url.rstrip("/")
        self.model = model
        self.api_key = api_key
        self.timeout_seconds = timeout_seconds

    def complete_json_sync(
        self,
        messages: list[dict],
        temperature: float = 0.0,
        max_tokens: int = 2048,
    ) -> dict:
        """Call `/chat/completions` and parse the assistant JSON object."""

        payload = {
            "model": self.model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "response_format": {"type": "json_object"},
        }
        data = json.dumps(payload).encode("utf-8")
        request = urllib.request.Request(
            f"{self.base_url}/chat/completions",
            data=data,
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {self.api_key}",
            },
            method="POST",
        )
        with urllib.request.urlopen(request, timeout=self.timeout_seconds) as response:
            raw = json.loads(response.read().decode("utf-8"))
        content = raw["choices"][0]["message"].get("content") or ""
        return _parse_json_object(content)


class TransformersJsonClient:
    """Direct Transformers JSON client for Kaggle GPU notebooks."""

    def __init__(self, model_name: str) -> None:
        """Load tokenizer and causal LM directly with Transformers."""

        try:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
        except Exception as exc:
            raise ImportError(
                "EXACT_LLM_PROVIDER='local' requires torch and transformers."
            ) from exc

        self.torch = torch
        print(f"Loading local tokenizer: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model_kwargs = {"torch_dtype": dtype}
        if importlib.util.find_spec("accelerate") is not None:
            model_kwargs["device_map"] = "auto"

        print(f"Loading local model: {model_name} with {model_kwargs}")
        started_at = time.monotonic()
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        if "device_map" not in model_kwargs and torch.cuda.is_available():
            self.model = self.model.to("cuda")
        print(f"Loaded local model in {time.monotonic() - started_at:.1f}s")

    def complete_json_sync(
        self,
        messages: list[dict],
        temperature: float = 0.0,
        max_tokens: int = 2048,
    ) -> dict:
        """Generate one response locally and parse the JSON object."""

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        input_length = inputs["input_ids"].shape[1]
        do_sample = temperature > 0
        generate_kwargs = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs.get("attention_mask"),
            "max_new_tokens": max_tokens,
            "max_time": LLM_TIMEOUT_SECONDS,
            "do_sample": do_sample,
            "use_cache": True,
            "pad_token_id": self.tokenizer.eos_token_id,
        }
        if do_sample:
            generate_kwargs["temperature"] = temperature

        started_at = time.monotonic()
        outputs = self.model.generate(**generate_kwargs)
        generated_tokens = outputs[0][input_length:]
        print(
            "Local generation finished: "
            f"tokens={len(generated_tokens)} "
            f"elapsed={time.monotonic() - started_at:.1f}s"
        )
        response_text = self.tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        )
        return _parse_json_object(response_text)


def build_json_client() -> object | None:
    """Build a JSON client for direct Transformers or OpenAI-compatible mode."""

    if not USE_LLM or LLM_PROVIDER == "none":
        return None
    if LLM_PROVIDER == "local":
        return TransformersJsonClient(LLM_MODEL)
    if LLM_PROVIDER == "openai":
        if not LLM_BASE_URL:
            raise ValueError("EXACT_LLM_BASE_URL is required for openai mode")
        return OpenAICompatibleJsonClient(
            base_url=LLM_BASE_URL,
            model=LLM_MODEL,
            api_key=LLM_API_KEY,
            timeout_seconds=LLM_TIMEOUT_SECONDS,
        )
    raise ValueError("EXACT_LLM_PROVIDER must be none, local, or openai")


def translate_with_fallback(
    premises: list[str],
    question: str,
    options: list[tuple[str, str]] | None = None,
    llm_client: object | None = None,
    allow_heuristic_fallback: bool = True,
) -> TranslationResult:
    """Try LLM translation first, then fall back to the local parser."""

    if llm_client is not None:
        try:
            return translate_with_llm(premises, question, options or [], llm_client)
        except Exception as exc:
            if not allow_heuristic_fallback:
                raise RuntimeError(
                    f"LLM translation failed and fallback is disabled: {exc}"
                ) from exc
            warnings = (f"LLM translation failed; heuristic parser used: {exc}",)
            return _heuristic_translation(premises, question, options or [], warnings)

    if not allow_heuristic_fallback:
        raise RuntimeError("No LLM client configured and fallback is disabled")

    return _heuristic_translation(
        premises,
        question,
        options or [],
        ("No LLM client configured; heuristic parser used.",),
    )


def translate_with_llm(
    premises: list[str],
    question: str,
    options: list[tuple[str, str]],
    llm_client: object,
) -> TranslationResult:
    """Translate Type 1 text into solver IR with a local LLM."""

    messages = _build_translation_messages(premises, question, options)
    raw = llm_client.complete_json_sync(
        messages=messages,
        temperature=LLM_TEMPERATURE,
        max_tokens=LLM_MAX_TOKENS,
    )
    normalized = _normalize_translation_payload(raw, premises, question, options)
    return _translation_payload_to_ir(normalized, premises, question)


def _heuristic_translation(
    premises: list[str],
    question: str,
    options: list[tuple[str, str]],
    warnings: tuple[str, ...] = (),
) -> TranslationResult:
    """Translate with local heuristics when LLM is unavailable or invalid."""

    parsed = tuple(parse_premise_to_ir(premise, idx) for idx, premise in enumerate(premises))
    option_atoms = {label: atom_from_text(text) for label, text in options}
    return TranslationResult(
        parsed_premises=parsed,
        query=parse_question_to_query(question),
        warnings=warnings,
        option_atoms=option_atoms,
    )


def _build_translation_messages(
    premises: list[str],
    question: str,
    options: list[tuple[str, str]],
) -> list[dict]:
    """Build a compact autoformalization prompt for a local <=8B LLM."""

    premise_text = "\n".join(f"{idx}: {premise}" for idx, premise in enumerate(premises))
    option_text = "\n".join(f"{label}. {text}" for label, text in options) or "None"
    schema_hint = (
        '{"predicates":[{"name":"p","arity":1,"gloss":"...",'
        '"argument_roles":["entity"]}],'
        '"premises":[{"source_idx":0,"facts":[ATOM],'
        '"rules":[{"conditions":[ATOM],"conclusion":ATOM}]}],'
        '"query":{"claim":ATOM},'
        '"options":[{"label":"A","text":"...","goal":ATOM}]}'
    )
    atom_hint = (
        'ATOM={"text":"short source phrase","pred":"snake_case",'
        '"args":["?x"],"negated":false}'
    )
    return [
        {
            "role": "system",
            "content": (
                "You are an autoformalizer for educational logic QA. "
                "Return JSON only. Do not answer the question. "
                "Translate text into Horn-style predicates for a symbolic solver."
            ),
        },
        {
            "role": "user",
            "content": (
                "Formalize premises, query, and MCQ options as compact JSON.\n"
                "Rules:\n"
                "- Output valid JSON only, matching this shape: "
                f"{schema_hint}\n"
                f"- {atom_hint}\n"
                "- Reuse predicate names from predicates everywhere; never invent variants.\n"
                "- pred and constants must be lowercase snake_case; variables use ?x, ?y.\n"
                "- Generic rules use variables; named facts/goals use constants.\n"
                "- Split conjunctions into separate condition atoms.\n"
                "- Preserve source_idx exactly.\n"
                "- Translate each A-D option into options[].goal when options exist.\n"
                "- Mark negated=true only for explicit negation.\n"
                "- Keep text fields short; do not include explanations.\n\n"
                f"Premises:\n{premise_text}\n\n"
                f"Question:\n{question}\n\n"
                f"Options:\n{option_text}"
            ),
        },
    ]


def _normalize_translation_payload(
    raw: dict,
    premises: list[str],
    question: str,
    options: list[tuple[str, str]],
) -> dict:
    """Repair common local-model JSON shape drift before IR conversion."""

    payload = _unwrap_payload(raw)
    premise_items = payload.get("premises") or payload.get("premise") or []
    if isinstance(premise_items, dict):
        premise_items = list(premise_items.values())
    if not isinstance(premise_items, list):
        premise_items = []

    normalized_premises = []
    used_indices: set[int] = set()
    for fallback_idx, item in enumerate(premise_items):
        if not isinstance(item, dict):
            continue
        source_idx = _safe_int(
            item.get("source_idx", item.get("idx", item.get("index", fallback_idx))),
            fallback_idx,
        )
        if source_idx < 0 or source_idx >= len(premises):
            continue
        used_indices.add(source_idx)
        facts = [_normalize_atom(atom) for atom in _as_list(item.get("facts") or item.get("fact"))]
        rules = [_normalize_rule(rule) for rule in _as_list(item.get("rules") or item.get("rule"))]
        facts = [atom for atom in facts if atom]
        rules = [rule for rule in rules if rule]
        if not facts and not rules and _looks_like_atom_dict(item):
            facts = [_normalize_atom(item)]
        normalized_premises.append(
            {"source_idx": source_idx, "facts": facts, "rules": rules}
        )

    for source_idx in range(len(premises)):
        if source_idx not in used_indices:
            normalized_premises.append({"source_idx": source_idx, "facts": [], "rules": []})

    query_payload = payload.get("query") or payload.get("question") or {}
    if isinstance(query_payload, dict):
        claim_payload = query_payload.get("claim") or query_payload.get("atom") or query_payload
    else:
        claim_payload = {"text": str(query_payload)}
    query = {"claim": _normalize_atom(claim_payload) or _normalize_atom({"text": question})}

    option_payloads = payload.get("options") or payload.get("choices") or []
    normalized_options = _normalize_option_payloads(option_payloads, options)

    return {"premises": normalized_premises, "query": query, "options": normalized_options}


def _translation_payload_to_ir(
    payload: dict,
    raw_premises: list[str],
    raw_question: str,
) -> TranslationResult:
    """Convert normalized translation JSON to internal IR objects."""

    parsed_by_idx: dict[int, ParsedPremise] = {}
    for premise_spec in payload["premises"]:
        source_idx = premise_spec["source_idx"]
        facts = tuple(
            Fact(
                atom=_atom_from_spec(atom_spec),
                source_idx=source_idx,
                text=raw_premises[source_idx],
            )
            for atom_spec in premise_spec.get("facts", [])
        )
        rules = tuple(
            Rule(
                conditions=tuple(
                    _atom_from_spec(atom_spec)
                    for atom_spec in rule_spec.get("conditions", [])
                ),
                conclusion=_atom_from_spec(rule_spec["conclusion"]),
                source_idx=source_idx,
                text=raw_premises[source_idx],
            )
            for rule_spec in premise_spec.get("rules", [])
            if rule_spec.get("conditions") and rule_spec.get("conclusion")
        )
        parsed_by_idx[source_idx] = ParsedPremise(facts=facts, rules=rules)

    parsed = tuple(
        parsed_by_idx.get(idx)
        or ParsedPremise(warnings=(f"No LLM IR for premise {idx}",))
        for idx in range(len(raw_premises))
    )
    query = Query(
        claim=_atom_from_spec(payload["query"]["claim"]),
        raw_question=raw_question,
    )
    option_atoms = {
        item["label"]: _atom_from_spec(item["goal"])
        for item in payload.get("options", [])
        if item.get("label") and item.get("goal")
    }
    return TranslationResult(parsed_premises=parsed, query=query, option_atoms=option_atoms)


def _atom_from_spec(spec: dict | None) -> Atom:
    """Convert normalized atom JSON to solver IR."""

    if spec is None:
        raise ValueError("atom spec is missing")

    pred = _slugify(str(spec.get("pred") or spec.get("predicate") or ""))
    args = tuple(_normalize_arg(arg) for arg in _as_list(spec.get("args") or spec.get("arguments")))
    args = tuple(arg for arg in args if arg)
    negated = bool(spec.get("negated") or spec.get("not") or spec.get("negative"))
    text = str(spec.get("text") or spec.get("source") or pred or "").strip()
    if pred:
        return Atom(pred=pred, args=args, negated=negated, text=text or None)
    atom = atom_from_text(text)
    return Atom(pred=atom.pred, args=atom.args, negated=negated or atom.negated, text=atom.text)


def _normalize_rule(rule: object) -> dict | None:
    """Normalize common rule key variants into conditions/conclusion."""

    if not isinstance(rule, dict):
        return None
    condition_items = (
        rule.get("conditions")
        or rule.get("condition")
        or rule.get("antecedent")
        or rule.get("if")
        or []
    )
    conclusion_item = rule.get("conclusion") or rule.get("consequent") or rule.get("then")
    conditions = [_normalize_atom(item) for item in _as_list(condition_items)]
    conditions = [item for item in conditions if item]
    conclusion = _normalize_atom(conclusion_item)
    if not conditions or not conclusion:
        return None
    return {"conditions": conditions, "conclusion": conclusion}


def _normalize_atom(atom: object) -> dict | None:
    """Normalize string or dict atom shapes into text/pred/args/negated."""

    if atom is None:
        return None
    if isinstance(atom, str):
        parsed = atom_from_text(atom)
        return {
            "text": atom,
            "pred": parsed.pred,
            "args": list(parsed.args),
            "negated": parsed.negated,
        }
    if not isinstance(atom, dict):
        return None

    text = str(
        atom.get("text")
        or atom.get("source")
        or atom.get("sentence")
        or atom.get("claim")
        or ""
    ).strip()
    pred = atom.get("pred") or atom.get("predicate") or atom.get("name")
    args = atom.get("args") or atom.get("arguments") or atom.get("entities") or []
    negated = bool(atom.get("negated") or atom.get("not") or atom.get("negative"))

    if not pred and text:
        parsed = atom_from_text(text)
        pred = parsed.pred
        args = list(parsed.args)
        negated = negated or parsed.negated
    if not pred:
        return None

    return {
        "text": text or str(pred),
        "pred": _slugify(str(pred)),
        "args": [_normalize_arg(arg) for arg in _as_list(args)],
        "negated": negated,
    }


def _normalize_option_payloads(
    option_payloads: object,
    fallback_options: list[tuple[str, str]],
) -> list[dict]:
    """Normalize LLM option goals, falling back to heuristic option atoms."""

    normalized: dict[str, dict] = {}
    for item in _as_list(option_payloads):
        if isinstance(item, dict):
            label = str(item.get("label") or item.get("option") or "").strip().upper()
            goal = _normalize_atom(
                item.get("goal") or item.get("claim") or item.get("atom") or item
            )
        else:
            label = ""
            goal = None
        if label in _OPTION_LABELS and goal:
            normalized[label] = {"label": label, "goal": goal}

    for label, text in fallback_options:
        if label not in normalized:
            parsed = atom_from_text(text)
            normalized[label] = {
                "label": label,
                "goal": {
                    "text": text,
                    "pred": parsed.pred,
                    "args": list(parsed.args),
                    "negated": parsed.negated,
                },
            }
    return [normalized[label] for label in sorted(normalized)]


def _parse_json_object(text: str) -> dict:
    """Extract and parse the first JSON object from model text."""

    text = str(text).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"LLM output did not contain a complete JSON object: {text[:300]}")
    parsed = json.loads(text[start : end + 1])
    if not isinstance(parsed, dict):
        raise ValueError("LLM JSON output must be an object")
    return parsed


def _unwrap_payload(raw: dict) -> dict:
    """Unwrap common top-level containers emitted by local models."""

    payload = raw
    for key in ("translation", "result", "output", "data"):
        if isinstance(payload, dict) and isinstance(payload.get(key), dict):
            payload = payload[key]
    return payload if isinstance(payload, dict) else {}


def _as_list(value: object) -> list:
    """Return a list for scalar, tuple, list, or missing JSON values."""

    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    return [value]


def _safe_int(value: object, default: int) -> int:
    """Parse an integer safely with fallback."""

    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _normalize_arg(value: object) -> str:
    """Normalize constants and variables for atom arguments."""

    text = str(value).strip().lower()
    if not text:
        return ""
    if text.startswith("?"):
        return "?" + _slugify(text[1:])
    return _slugify(text)


def _looks_like_atom_dict(value: dict) -> bool:
    """Return whether a dict has atom-like keys."""

    return bool({"text", "pred", "predicate", "args", "arguments"} & set(value))

## 8. Type 1 Pipeline

Pipeline này chạy LLM-first autoformalization rồi symbolic reasoning. Với MCQ, notebook cũng cho LLM canonicalize từng option nếu model trả field `options[].goal`; nếu không có thì fallback option parser.

In [23]:
_OPTION_RE = re.compile(
    r"(?:^|\n)\s*([A-D])\.\s*(.*?)(?=(?:\n\s*[A-D]\.\s*)|\Z)",
    re.IGNORECASE | re.DOTALL,
)
_OPTION_ORDER = ("A", "B", "C", "D")


def extract_options(question: str) -> list[tuple[str, str]]:
    """Extract A-D option labels and text from a multiple-choice question."""

    options = [
        (match.group(1).upper(), " ".join(match.group(2).split()))
        for match in _OPTION_RE.finditer(question)
    ]
    seen: set[str] = set()
    deduped: list[tuple[str, str]] = []
    for label, text in options:
        if label in seen or not text:
            continue
        seen.add(label)
        deduped.append((label, text))
    return deduped


def strip_options_from_question(question: str) -> str:
    """Return the question stem before the first MCQ option."""

    first_option = _OPTION_RE.search(question)
    stem = question[: first_option.start()] if first_option else question
    return " ".join(stem.split())


def build_goals_for_mcq(
    options: list[tuple[str, str]],
    option_atoms: dict[str, Atom] | None = None,
) -> list[tuple[str, Atom]]:
    """Build one solver goal per MCQ option."""

    option_atoms = option_atoms or {}
    return [(label, option_atoms.get(label) or atom_from_text(text)) for label, text in options]


def evaluate_mcq_options(
    kb: KnowledgeBase,
    goals: list[tuple[str, Atom]],
    solver: ForwardChainSolver | None = None,
) -> dict[str, SolveResult]:
    """Run the symbolic solver independently for each option."""

    solver = solver or ForwardChainSolver()
    return {label: solver.solve(kb, goal) for label, goal in goals}


def decide_mcq_winner(results: dict[str, SolveResult], question_stem: str) -> str:
    """Select the best option from entailment labels and proof size."""

    if not results:
        return "A"

    ordered_labels = [label for label in _OPTION_ORDER if label in results]
    entailed = [label for label in ordered_labels if results[label].label == "Yes"]
    if len(entailed) == 1:
        return entailed[0]

    candidates = entailed or ordered_labels
    rank = {"Yes": 0, "Unknown": 1, "No": 2}

    def score(label: str) -> tuple[int, int, int]:
        """Rank entailed labels before unknown/no, then shorter proofs."""

        result = results[label]
        proof_size = len(result.supporting_premises) if result.supporting_premises else 999
        return rank.get(result.label, 3), proof_size, _OPTION_ORDER.index(label)

    if entailed and "fewest premises" in question_stem.lower():
        return min(
            entailed,
            key=lambda label: (
                len(results[label].supporting_premises),
                _OPTION_ORDER.index(label),
            ),
        )

    return min(candidates, key=score)


def run_type1_pipeline(
    request: PredictionRequest,
    translator_client: object | None = None,
    allow_heuristic_fallback: bool = True,
    question_type: QuestionType | None = None,
) -> PredictionResponse:
    """Answer a Type 1 logic query with symbolic proof where possible."""

    routed_question_type = question_type or QuestionType.YES_NO_UNCERTAIN
    premises = request.premises_nl or []
    options = extract_options(request.question) if routed_question_type == QuestionType.MCQ else []
    translation = translate_with_fallback(
        premises=premises,
        question=request.question,
        options=options,
        llm_client=translator_client,
        allow_heuristic_fallback=allow_heuristic_fallback,
    )
    parser_version = (
        "llm_translator_inline_v1"
        if translator_client is not None
        else "heuristic_horn_v1"
    )
    kb = build_kb_from_parsed_premises(
        premises,
        translation.parsed_premises,
        parser_version=parser_version,
        extra_warnings=translation.warnings,
    )

    if routed_question_type == QuestionType.MCQ:
        return _run_mcq_path(request, kb, routed_question_type, translation.option_atoms)

    result = ForwardChainSolver().solve(kb, translation.query.claim)
    explanation, cot, cited_premises = explain_result(result, kb)
    confidence = {"Yes": 0.78, "No": 0.76, "Unknown": 0.35}[result.label]

    return PredictionResponse(
        id=request.id,
        task_type=TaskType.TYPE1_LOGIC,
        question_type=routed_question_type,
        answer=result.label,
        explanation=explanation,
        fol=kb_to_fol_like_text(kb) or None,
        cot=cot,
        premises=cited_premises,
        confidence=confidence,
        error="; ".join(result.warnings) if result.warnings else None,
    )


def _run_mcq_path(
    request: PredictionRequest,
    kb: KnowledgeBase,
    question_type: QuestionType,
    option_atoms: dict[str, Atom] | None = None,
) -> PredictionResponse:
    """Evaluate and select a multiple-choice option."""

    options = extract_options(request.question)
    if not options:
        return PredictionResponse(
            id=request.id,
            task_type=TaskType.TYPE1_LOGIC,
            question_type=question_type,
            answer="A",
            explanation=(
                "No multiple-choice options were parsed, so the system returned "
                "the default option A."
            ),
            fol=kb_to_fol_like_text(kb) or None,
            cot=["No option labels A-D were available for symbolic evaluation."],
            premises=[],
            confidence=0.05,
            error="MCQ routed but no options were parsed.",
        )

    stem = strip_options_from_question(request.question)
    goals = build_goals_for_mcq(options, option_atoms=option_atoms)
    results = evaluate_mcq_options(kb, goals)
    winner = decide_mcq_winner(results, stem)
    winning_result = results[winner]
    explanation, proof_cot, cited_premises = explain_result(winning_result, kb)
    if winning_result.label == "Unknown":
        explanation = "The selected option has the best symbolic ranking, but no proof was found."
    option_summary = _format_mcq_option_summary(results)
    no_entailed_option = all(result.label != "Yes" for result in results.values())

    return PredictionResponse(
        id=request.id,
        task_type=TaskType.TYPE1_LOGIC,
        question_type=question_type,
        answer=winner,
        explanation=f"Option {winner} is selected. {explanation} {option_summary}",
        fol=kb_to_fol_like_text(kb) or None,
        cot=[*proof_cot, option_summary],
        premises=cited_premises,
        confidence=_mcq_confidence(results, winner),
        error="No MCQ option was symbolically entailed." if no_entailed_option else None,
    )


def _format_mcq_option_summary(results: dict[str, SolveResult]) -> str:
    """Format each option entailment label for debugging."""

    parts = [f"{label}: {results[label].label}" for label in _OPTION_ORDER if label in results]
    return "Option entailment results: " + ", ".join(parts) + "."


def _mcq_confidence(results: dict[str, SolveResult], winner: str) -> float:
    """Return a conservative confidence from option entailment structure."""

    if results[winner].label == "Yes":
        entailed_count = sum(result.label == "Yes" for result in results.values())
        return 0.72 if entailed_count == 1 else 0.58
    if results[winner].label == "Unknown":
        return 0.28
    return 0.12

## 8.1 Premise-Level LLM Cache Override

Cell này override pipeline ở trên để tránh gọi LLM dịch lại toàn bộ premise cho từng question. Ý tưởng:

1. Dịch `premises-NL` một lần theo `premise_hash`.
2. Build/cache `KnowledgeBase` từ bản dịch đó.
3. Với mỗi question, chỉ gọi LLM ngắn để dịch `query` và `options` dựa trên predicate inventory đã có.

Đây là tối ưu quan trọng cho Kaggle/T4 vì premise groups được dùng lại nhiều lần, còn gọi Qwen 7B để generate toàn bộ JSON premises cho mỗi instance sẽ vượt xa 60s/request.

In [24]:
TRANSLATED_KB_CACHE: dict[str, tuple[KnowledgeBase, tuple[str, ...]]] = {}


def _predicate_inventory_from_kb(kb: KnowledgeBase) -> list[str]:
    """Return compact predicate names available to query/option translation."""

    names = {fact.atom.pred for fact in kb.facts}
    for rule in kb.rules:
        names.add(rule.conclusion.pred)
        names.update(condition.pred for condition in rule.conditions)
    return sorted(names)




def _premise_payload_to_parsed(
    payload: dict,
    raw_premises: list[str],
) -> tuple[ParsedPremise, ...]:
    """Convert normalized premise-only JSON without requiring a query claim."""

    parsed_by_idx: dict[int, ParsedPremise] = {}
    for premise_spec in payload.get("premises", []):
        source_idx = premise_spec.get("source_idx")
        if source_idx is None or source_idx < 0 or source_idx >= len(raw_premises):
            continue

        facts = tuple(
            Fact(
                atom=_atom_from_spec(atom_spec),
                source_idx=source_idx,
                text=raw_premises[source_idx],
            )
            for atom_spec in premise_spec.get("facts", [])
            if atom_spec is not None
        )
        rules = tuple(
            Rule(
                conditions=tuple(
                    _atom_from_spec(atom_spec)
                    for atom_spec in rule_spec.get("conditions", [])
                    if atom_spec is not None
                ),
                conclusion=_atom_from_spec(rule_spec.get("conclusion")),
                source_idx=source_idx,
                text=raw_premises[source_idx],
            )
            for rule_spec in premise_spec.get("rules", [])
            if rule_spec.get("conditions") and rule_spec.get("conclusion")
        )
        parsed_by_idx[source_idx] = ParsedPremise(facts=facts, rules=rules)

    return tuple(
        parsed_by_idx.get(idx)
        or ParsedPremise(warnings=(f"No LLM IR for premise {idx}",))
        for idx in range(len(raw_premises))
    )

def translate_premises_with_fallback(
    premises: list[str],
    llm_client: object | None,
    allow_heuristic_fallback: bool,
) -> tuple[tuple[ParsedPremise, ...], tuple[str, ...]]:
    """Translate premises only; this result is safe to cache per premise hash."""

    if llm_client is not None:
        try:
            messages = _build_premise_translation_messages(premises)
            raw = llm_client.complete_json_sync(
                messages=messages,
                temperature=LLM_TEMPERATURE,
                max_tokens=LLM_MAX_TOKENS,
            )
            normalized = _normalize_translation_payload(raw, premises, "", [])
            parsed_premises = _premise_payload_to_parsed(normalized, premises)
            return parsed_premises, ()
        except Exception as exc:
            if not allow_heuristic_fallback:
                raise RuntimeError(
                    f"LLM premise translation failed and fallback is disabled: {exc}"
                ) from exc
            warnings = (f"LLM premise translation failed; heuristic parser used: {exc}",)
            parsed = tuple(
                parse_premise_to_ir(premise, idx)
                for idx, premise in enumerate(premises)
            )
            return parsed, warnings

    if not allow_heuristic_fallback:
        raise RuntimeError("No LLM client configured and fallback is disabled")

    parsed = tuple(
        parse_premise_to_ir(premise, idx)
        for idx, premise in enumerate(premises)
    )
    return parsed, ("No LLM client configured; heuristic parser used.",)


def get_or_translate_kb(
    premises: list[str],
    llm_client: object | None,
    allow_heuristic_fallback: bool,
) -> tuple[KnowledgeBase, tuple[str, ...]]:
    """Return a KB cached after premise-level LLM translation."""

    key = hash_premises(tuple(premises), parser_version="llm_premise_cache_v1")
    cached = TRANSLATED_KB_CACHE.get(key)
    if cached is not None:
        return cached

    parsed_premises, warnings = translate_premises_with_fallback(
        premises=premises,
        llm_client=llm_client,
        allow_heuristic_fallback=allow_heuristic_fallback,
    )
    kb = build_kb_from_parsed_premises(
        premises=premises,
        parsed_premises=parsed_premises,
        parser_version="llm_premise_cache_v1" if llm_client is not None else "heuristic_horn_v1",
        extra_warnings=warnings,
    )
    TRANSLATED_KB_CACHE[key] = (kb, warnings)
    return kb, warnings


def translate_query_with_fallback(
    question: str,
    options: list[tuple[str, str]],
    kb: KnowledgeBase,
    llm_client: object | None,
    allow_heuristic_fallback: bool,
) -> tuple[Query, dict[str, Atom], tuple[str, ...]]:
    """Translate only query/options using the cached KB predicate inventory."""

    if llm_client is not None:
        try:
            messages = _build_query_translation_messages(
                question,
                options,
                _predicate_inventory_from_kb(kb),
            )
            raw = llm_client.complete_json_sync(
                messages=messages,
                temperature=LLM_TEMPERATURE,
                max_tokens=min(LLM_MAX_TOKENS, 256),
            )
            normalized = _normalize_query_payload(raw, question, options)
            query = Query(
                claim=_atom_from_spec(normalized["query"]["claim"]),
                raw_question=question,
            )
            option_atoms = {
                item["label"]: _atom_from_spec(item["goal"])
                for item in normalized.get("options", [])
                if item.get("label") and item.get("goal")
            }
            return query, option_atoms, ()
        except Exception as exc:
            if not allow_heuristic_fallback:
                raise RuntimeError(
                    f"LLM query translation failed and fallback is disabled: {exc}"
                ) from exc
            warnings = (f"LLM query translation failed; heuristic parser used: {exc}",)
            option_atoms = {label: atom_from_text(text) for label, text in options}
            return parse_question_to_query(question), option_atoms, warnings

    if not allow_heuristic_fallback:
        raise RuntimeError("No LLM client configured and fallback is disabled")

    return (
        parse_question_to_query(question),
        {label: atom_from_text(text) for label, text in options},
        ("No LLM client configured; heuristic parser used.",),
    )


def _build_premise_translation_messages(premises: list[str]) -> list[dict]:
    """Build a compact prompt that formalizes premises only."""

    premise_text = "\n".join(f"{idx}: {premise}" for idx, premise in enumerate(premises))
    return [
        {
            "role": "system",
            "content": (
                "You are an autoformalizer for educational logic QA. "
                "Return minified JSON only. Do not answer questions."
            ),
        },
        {
            "role": "user",
            "content": (
                "Formalize only the premises as Horn facts/rules. "
                "Use this JSON shape: {\"predicates\":[{\"name\":\"p\","
                "\"arity\":1,\"gloss\":\"...\","
                "\"argument_roles\":[\"entity\"]}],"
                "\"premises\":[{\"source_idx\":0,\"facts\":[ATOM],"
                "\"rules\":[{\"conditions\":[ATOM],"
                "\"conclusion\":ATOM}]}],"
                "\"query\":{\"claim\":{\"text\":\"placeholder\","
                "\"pred\":\"placeholder\",\"args\":[],"
                "\"negated\":false}},\"options\":[]}. "
                "ATOM={\"text\":\"short\",\"pred\":\"snake_case\","
                "\"args\":[\"?x\"],\"negated\":false}. "
                "Reuse predicate names, split conjunctions, use ?x for generic "
                "rules, lowercase constants for names.\n"
                f"Premises:\n{premise_text}"
            ),
        },
    ]


def _build_query_translation_messages(
    question: str,
    options: list[tuple[str, str]],
    predicates: list[str],
) -> list[dict]:
    """Build a short prompt that formalizes query/options only."""

    option_text = "\n".join(f"{label}. {text}" for label, text in options) or "None"
    predicate_text = ", ".join(predicates[:80])
    return [
        {
            "role": "system",
            "content": "Return minified JSON only. Translate query/options to existing predicates.",
        },
        {
            "role": "user",
            "content": (
                "Use only these predicate names when possible: "
                f"{predicate_text}. "
                "Return {\"query\":{\"claim\":ATOM},\"options\":[{"
                "\"label\":\"A\",\"text\":\"...\",\"goal\":ATOM}]}. "
                "ATOM={\"text\":\"short\",\"pred\":\"snake_case\","
                "\"args\":[\"sophia\"],\"negated\":false}. "
                "Do not answer the question.\n"
                f"Question:\n{question}\nOptions:\n{option_text}"
            ),
        },
    ]


def _normalize_query_payload(
    raw: dict,
    question: str,
    options: list[tuple[str, str]],
) -> dict:
    """Normalize query-only JSON into the same query/options subset."""

    payload = _unwrap_payload(raw)
    query_payload = payload.get("query") or payload.get("question") or {}
    if isinstance(query_payload, dict):
        claim_payload = query_payload.get("claim") or query_payload.get("atom") or query_payload
    else:
        claim_payload = {"text": str(query_payload)}
    query = {"claim": _normalize_atom(claim_payload) or _normalize_atom({"text": question})}
    option_payloads = payload.get("options") or payload.get("choices") or []
    return {"query": query, "options": _normalize_option_payloads(option_payloads, options)}


# Override the earlier Type 1 pipeline with the cached LLM-premise version.
def run_type1_pipeline(
    request: PredictionRequest,
    translator_client: object | None = None,
    allow_heuristic_fallback: bool = True,
    question_type: QuestionType | None = None,
) -> PredictionResponse:
    """Answer Type 1 with cached premise translation plus lightweight query translation."""

    routed_question_type = question_type or QuestionType.YES_NO_UNCERTAIN
    premises = request.premises_nl or []
    options = extract_options(request.question) if routed_question_type == QuestionType.MCQ else []
    kb, kb_warnings = get_or_translate_kb(
        premises=premises,
        llm_client=translator_client,
        allow_heuristic_fallback=allow_heuristic_fallback,
    )
    query, option_atoms, query_warnings = translate_query_with_fallback(
        question=request.question,
        options=options,
        kb=kb,
        llm_client=translator_client,
        allow_heuristic_fallback=allow_heuristic_fallback,
    )
    if kb_warnings or query_warnings:
        kb = dataclasses.replace(kb, warnings=tuple((*kb.warnings, *query_warnings)))

    if routed_question_type == QuestionType.MCQ:
        return _run_mcq_path(request, kb, routed_question_type, option_atoms)

    result = ForwardChainSolver().solve(kb, query.claim)
    explanation, cot, cited_premises = explain_result(result, kb)
    confidence = {"Yes": 0.78, "No": 0.76, "Unknown": 0.35}[result.label]
    return PredictionResponse(
        id=request.id,
        task_type=TaskType.TYPE1_LOGIC,
        question_type=routed_question_type,
        answer=result.label,
        explanation=explanation,
        fol=kb_to_fol_like_text(kb) or None,
        cot=cot,
        premises=cited_premises,
        confidence=confidence,
        error="; ".join(result.warnings) if result.warnings else None,
    )

In [25]:
def _premise_payload_to_parsed_premises(
    payload: dict,
    raw_premises: list[str],
) -> tuple[ParsedPremise, ...]:
    """Convert premise-only translation JSON without requiring a query claim."""

    parsed_by_idx: dict[int, ParsedPremise] = {}
    for premise_spec in payload.get("premises", []):
        source_idx = premise_spec.get("source_idx")
        if not isinstance(source_idx, int) or source_idx < 0 or source_idx >= len(raw_premises):
            continue
        facts = tuple(
            Fact(
                atom=_atom_from_spec(atom_spec),
                source_idx=source_idx,
                text=raw_premises[source_idx],
            )
            for atom_spec in premise_spec.get("facts", [])
            if atom_spec
        )
        rules = tuple(
            Rule(
                conditions=tuple(
                    _atom_from_spec(atom_spec)
                    for atom_spec in rule_spec.get("conditions", [])
                    if atom_spec
                ),
                conclusion=_atom_from_spec(rule_spec["conclusion"]),
                source_idx=source_idx,
                text=raw_premises[source_idx],
            )
            for rule_spec in premise_spec.get("rules", [])
            if rule_spec.get("conditions") and rule_spec.get("conclusion")
        )
        parsed_by_idx[source_idx] = ParsedPremise(facts=facts, rules=rules)

    return tuple(
        parsed_by_idx.get(idx)
        or ParsedPremise(warnings=(f"No LLM IR for premise {idx}",))
        for idx in range(len(raw_premises))
    )


# Patch premise-only translation: the premise cache stage does not need query/options.
def translate_premises_with_fallback(
    premises: list[str],
    llm_client: object | None,
    allow_heuristic_fallback: bool,
) -> tuple[tuple[ParsedPremise, ...], tuple[str, ...]]:
    """Translate premises only; this result is safe to cache per premise hash."""

    if llm_client is not None:
        try:
            messages = _build_premise_translation_messages(premises)
            raw = llm_client.complete_json_sync(
                messages=messages,
                temperature=LLM_TEMPERATURE,
                max_tokens=LLM_MAX_TOKENS,
            )
            normalized = _normalize_translation_payload(raw, premises, "placeholder", [])
            return _premise_payload_to_parsed_premises(normalized, premises), ()
        except Exception as exc:
            if not allow_heuristic_fallback:
                raise RuntimeError(
                    f"LLM premise translation failed and fallback is disabled: {exc}"
                ) from exc
            warnings = (f"LLM premise translation failed; heuristic parser used: {exc}",)
            parsed = tuple(parse_premise_to_ir(premise, idx) for idx, premise in enumerate(premises))
            return parsed, warnings

    if not allow_heuristic_fallback:
        raise RuntimeError("No LLM client configured and fallback is disabled")

    parsed = tuple(parse_premise_to_ir(premise, idx) for idx, premise in enumerate(premises))
    return parsed, ("No LLM client configured; heuristic parser used.",)


# Make atom conversion robust to null atoms from small local models.
_original_atom_from_spec = _atom_from_spec


def _atom_from_spec(spec: dict | None) -> Atom:
    """Convert normalized atom JSON to solver IR, tolerating null model output."""

    if not isinstance(spec, dict):
        return Atom(pred="unknown", args=(), text="unknown")
    return _original_atom_from_spec(spec)


In [26]:
# Patch query translation for small local models: malformed query JSON should not kill the batch.
def translate_query_with_fallback(
    question: str,
    options: list[tuple[str, str]],
    kb: KnowledgeBase,
    llm_client: object | None,
    allow_heuristic_fallback: bool,
) -> tuple[Query, dict[str, Atom], tuple[str, ...]]:
    """Translate query/options, falling back locally when Qwen emits invalid JSON."""

    heuristic_query = parse_question_to_query(question)
    heuristic_options = {label: atom_from_text(text) for label, text in options}

    if llm_client is not None:
        try:
            messages = _build_query_translation_messages(
                question,
                options,
                _predicate_inventory_from_kb(kb),
            )
            raw = llm_client.complete_json_sync(
                messages=messages,
                temperature=0.0,
                max_tokens=min(LLM_MAX_TOKENS, 128),
            )
            normalized = _normalize_query_payload(raw, question, options)
            query = Query(claim=_atom_from_spec(normalized["query"]["claim"]), raw_question=question)
            option_atoms = {
                item["label"]: _atom_from_spec(item["goal"])
                for item in normalized.get("options", [])
                if item.get("label") and item.get("goal")
            }
            return query, option_atoms or heuristic_options, ()
        except Exception as exc:
            warnings = (f"LLM query translation failed; heuristic query parser used: {exc}",)
            return heuristic_query, heuristic_options, warnings

    if not allow_heuristic_fallback:
        raise RuntimeError("No LLM client configured and fallback is disabled")

    return (
        heuristic_query,
        heuristic_options,
        ("No LLM client configured; heuristic query parser used.",),
    )


In [27]:
# Compact premise JSON patch for Qwen: fewer keys/braces means fewer malformed-JSON failures.
def _build_premise_translation_messages(premises: list[str]) -> list[dict]:
    """Build a compact premise-only prompt that Qwen can keep as valid JSON."""

    premise_text = "\n".join(f"{idx}: {premise}" for idx, premise in enumerate(premises))
    return [
        {
            "role": "system",
            "content": (
                "You convert educational premises to compact Horn logic JSON. "
                "Return exactly one minified JSON object and nothing else."
            ),
        },
        {
            "role": "user",
            "content": (
                "Use this compact schema only:\n"
                "{\"predicates\":[[name,arity,gloss]],"
                "\"facts\":[[source_idx,pred,args,negated]],"
                "\"rules\":[[source_idx,conditions,conclusion]]}\n"
                "Atom format: [pred,args,negated]. args is a list.\n"
                "Rule conditions is a list of Atom. Rule conclusion is one Atom.\n"
                "Use lowercase snake_case predicates and constants. Use ?x for generic rules.\n"
                "Split conjunctions into multiple condition atoms.\n"
                "No markdown. No explanations. No trailing commas.\n"
                "Example:\n"
                "{\"predicates\":[[\"completed_core_curriculum\",1,\"entity completed core curriculum\"],[\"passed_science_assessment\",1,\"entity passed science assessment\"],[\"qualified_for_advanced_courses\",1,\"entity qualifies for advanced courses\"]],"
                "\"facts\":[[1,\"completed_core_curriculum\",[\"sophia\"],false]],"
                "\"rules\":[[0,[[\"completed_core_curriculum\",[\"?x\"],false],[\"passed_science_assessment\",[\"?x\"],false]],[\"qualified_for_advanced_courses\",[\"?x\"],false]]]}\n"
                f"Premises:\n{premise_text}"
            ),
        },
    ]


def _atom_from_compact(item: object) -> Atom | None:
    """Convert compact [pred,args,negated] into Atom."""

    if not isinstance(item, list) or len(item) < 2:
        return None
    pred = _slugify(str(item[0]))
    args = tuple(_normalize_arg(arg) for arg in _as_list(item[1]))
    negated = bool(item[2]) if len(item) > 2 else False
    if not pred:
        return None
    return Atom(pred=pred, args=tuple(arg for arg in args if arg), negated=negated, text=pred)


def _premise_payload_to_parsed_premises(
    payload: dict,
    raw_premises: list[str],
) -> tuple[ParsedPremise, ...]:
    """Convert compact or nested premise-only JSON into ParsedPremise objects."""

    parsed_by_idx: dict[int, ParsedPremise] = {}

    compact_facts = payload.get("facts") if isinstance(payload, dict) else None
    compact_rules = payload.get("rules") if isinstance(payload, dict) else None
    if isinstance(compact_facts, list) or isinstance(compact_rules, list):
        facts_by_idx: dict[int, list[Fact]] = {}
        rules_by_idx: dict[int, list[Rule]] = {}
        for item in compact_facts or []:
            if not isinstance(item, list) or len(item) < 3:
                continue
            source_idx = _safe_int(item[0], -1)
            if source_idx < 0 or source_idx >= len(raw_premises):
                continue
            atom = _atom_from_compact([item[1], item[2], item[3] if len(item) > 3 else False])
            if atom is not None:
                facts_by_idx.setdefault(source_idx, []).append(
                    Fact(atom=atom, source_idx=source_idx, text=raw_premises[source_idx])
                )
        for item in compact_rules or []:
            if not isinstance(item, list) or len(item) < 3:
                continue
            source_idx = _safe_int(item[0], -1)
            if source_idx < 0 or source_idx >= len(raw_premises):
                continue
            conditions = tuple(
                atom for atom in (_atom_from_compact(condition) for condition in _as_list(item[1])) if atom
            )
            conclusion = _atom_from_compact(item[2])
            if conditions and conclusion:
                rules_by_idx.setdefault(source_idx, []).append(
                    Rule(
                        conditions=conditions,
                        conclusion=conclusion,
                        source_idx=source_idx,
                        text=raw_premises[source_idx],
                    )
                )
        for idx in range(len(raw_premises)):
            parsed_by_idx[idx] = ParsedPremise(
                facts=tuple(facts_by_idx.get(idx, [])),
                rules=tuple(rules_by_idx.get(idx, [])),
            )
        return tuple(parsed_by_idx[idx] for idx in range(len(raw_premises)))

    for premise_spec in payload.get("premises", []):
        source_idx = premise_spec.get("source_idx")
        if not isinstance(source_idx, int) or source_idx < 0 or source_idx >= len(raw_premises):
            continue
        facts = tuple(
            Fact(atom=_atom_from_spec(atom_spec), source_idx=source_idx, text=raw_premises[source_idx])
            for atom_spec in premise_spec.get("facts", [])
            if atom_spec
        )
        rules = tuple(
            Rule(
                conditions=tuple(
                    _atom_from_spec(atom_spec)
                    for atom_spec in rule_spec.get("conditions", [])
                    if atom_spec
                ),
                conclusion=_atom_from_spec(rule_spec["conclusion"]),
                source_idx=source_idx,
                text=raw_premises[source_idx],
            )
            for rule_spec in premise_spec.get("rules", [])
            if rule_spec.get("conditions") and rule_spec.get("conclusion")
        )
        parsed_by_idx[source_idx] = ParsedPremise(facts=facts, rules=rules)

    return tuple(
        parsed_by_idx.get(idx) or ParsedPremise(warnings=(f"No LLM IR for premise {idx}",))
        for idx in range(len(raw_premises))
    )


def translate_premises_with_fallback(
    premises: list[str],
    llm_client: object | None,
    allow_heuristic_fallback: bool,
) -> tuple[tuple[ParsedPremise, ...], tuple[str, ...]]:
    """Translate premises only using compact JSON; do not require query/options."""

    if llm_client is not None:
        try:
            messages = _build_premise_translation_messages(premises)
            raw = llm_client.complete_json_sync(
                messages=messages,
                temperature=0.0,
                max_tokens=min(LLM_MAX_TOKENS, 768),
            )
            return _premise_payload_to_parsed_premises(raw, premises), ()
        except Exception as exc:
            if not allow_heuristic_fallback:
                raise RuntimeError(
                    f"LLM premise translation failed and fallback is disabled: {exc}"
                ) from exc
            warnings = (f"LLM premise translation failed; heuristic parser used: {exc}",)
            parsed = tuple(parse_premise_to_ir(premise, idx) for idx, premise in enumerate(premises))
            return parsed, warnings

    if not allow_heuristic_fallback:
        raise RuntimeError("No LLM client configured and fallback is disabled")

    parsed = tuple(parse_premise_to_ir(premise, idx) for idx, premise in enumerate(premises))
    return parsed, ("No LLM client configured; heuristic parser used.",)


## 9. Router and Type 2 Placeholder

Router dùng shape của input để chọn Type 1/Type 2. Type 2 giữ placeholder để notebook không crash nếu batch có physics sample; bạn của bạn có thể thay function `run_type2_pipeline` trong cell này.

In [28]:
_OPTION_LABEL_RE = re.compile(r"(?:^|\n)\s*([A-D])\.\s+", re.IGNORECASE)
_YNU_STEM_RE = re.compile(
    r"^\s*(?:based on the above premises,\s*)?"
    r"(?:does|do|did|is|are|can|could|will|would|should)\b",
    re.IGNORECASE,
)
TYPE2_NOT_IMPLEMENTED_MESSAGE = (
    "Type 2 physics reasoning is reserved for the dedicated physics pipeline. "
    "This placeholder keeps the notebook contract stable while the team "
    "implements quantity extraction, formula selection, execution, and verification."
)


@dataclasses.dataclass(frozen=True)
class RouteDecision:
    """Routing decision with reason and question subtype."""

    task_type: TaskType
    reason: str
    question_type: QuestionType = QuestionType.UNKNOWN


class TaskRouter:
    """Route normalized requests to Type 1 logic or Type 2 physics."""

    def route(self, request: PredictionRequest) -> RouteDecision:
        """Return task and question type for one request."""

        if request.premises_nl:
            question_type = detect_question_type(request)
            return RouteDecision(
                task_type=TaskType.TYPE1_LOGIC,
                reason=f"premises_nl present; question_type={question_type.value}",
                question_type=question_type,
            )
        return RouteDecision(
            task_type=TaskType.TYPE2_PHYSICS,
            reason="premises_nl absent",
            question_type=QuestionType.NUMERICAL,
        )


def detect_question_type(request: PredictionRequest) -> QuestionType:
    """Detect MCQ, yes/no/unknown, or open-ended question shape."""

    question = request.question or ""
    labels = [match.group(1).upper() for match in _OPTION_LABEL_RE.finditer(question)]
    if {"A", "B", "C", "D"}.issubset(labels):
        return QuestionType.MCQ
    if _YNU_STEM_RE.search(question):
        return QuestionType.YES_NO_UNCERTAIN
    return QuestionType.OPEN_ENDED


def run_type2_pipeline(request: PredictionRequest) -> PredictionResponse:
    """Stable API placeholder for future Type 2 physics work."""

    return PredictionResponse(
        id=request.id,
        task_type=TaskType.TYPE2_PHYSICS,
        question_type=QuestionType.NUMERICAL,
        answer="",
        explanation=TYPE2_NOT_IMPLEMENTED_MESSAGE,
        fol=None,
        cot=[
            "The request was routed to the Type 2 physics branch.",
            "The production physics solver has not been implemented in this notebook yet.",
        ],
        premises=[
            "Type 2 receives only the question text.",
            "A future physics pipeline should derive answers from formulas and units.",
        ],
        confidence=0.0,
        error="type2_pipeline_not_implemented",
    )

## 10. Dataset Loading and Batch Runner

Cell này thay `scripts/run_predictions.py`. Nó tự tìm input trên Kaggle, chạy toàn batch, và lưu cả response nội bộ lẫn object `official` để dễ submit/evaluate.

In [29]:
def resolve_input_path() -> pathlib.Path:
    """Find the inference JSON path on Kaggle or in the local repo."""

    if INPUT_PATH:
        return pathlib.Path(INPUT_PATH)

    candidates = sorted(
        glob.glob("/kaggle/input/**/Logic_Based_Educational_Queries_inference.json", recursive=True)
    )
    if candidates:
        return pathlib.Path(candidates[0])

    local = pathlib.Path(LOCAL_FALLBACK_INPUT)
    if local.exists():
        return local

    raise FileNotFoundError(
        "Cannot find Logic_Based_Educational_Queries_inference.json. "
        "Set EXACT_INPUT_PATH to the dataset file."
    )


def load_instances(path: pathlib.Path) -> list[dict]:
    """Load a JSON prediction batch from a list or `instances` object."""

    payload = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(payload, list):
        instances = payload
    elif isinstance(payload, dict) and isinstance(payload.get("instances"), list):
        instances = payload["instances"]
    else:
        raise ValueError(f"Expected a list or a top-level instances list in {path}")

    if not all(isinstance(instance, dict) for instance in instances):
        raise ValueError(f"All instances in {path} must be JSON objects")
    return instances


def run_predictions(
    input_path: pathlib.Path,
    output_path: pathlib.Path,
    limit: int | None = None,
) -> dict:
    """Run the full EXACT prediction flow and write JSON output."""

    instances = load_instances(input_path)
    if limit is not None:
        instances = instances[:limit]

    router = TaskRouter()
    translator_client = build_json_client()
    predictions: list[dict] = []
    started_at = time.monotonic()
    print(
        f"Running {len(instances)} instances | "
        f"llm_enabled={translator_client is not None} | require_llm={REQUIRE_LLM}"
    )

    for index, instance in enumerate(instances, start=1):
        request = PredictionRequest.from_dict(instance)
        route = router.route(request)

        if route.task_type == TaskType.TYPE1_LOGIC:
            response = run_type1_pipeline(
                request,
                translator_client=translator_client,
                allow_heuristic_fallback=not REQUIRE_LLM,
                question_type=route.question_type,
            )
        elif route.task_type == TaskType.TYPE2_PHYSICS:
            response = run_type2_pipeline(request)
        else:
            raise ValueError(f"Unsupported task type: {route.task_type}")

        prediction = response.to_dict()
        prediction["route_reason"] = route.reason
        prediction["official"] = to_official_response(response)
        predictions.append(prediction)

        if PROGRESS_EVERY and index % PROGRESS_EVERY == 0:
            elapsed = time.monotonic() - started_at
            print(f"Processed {index}/{len(instances)} in {elapsed:.1f}s")

    output = {
        "source": str(input_path),
        "count": len(predictions),
        "format": "exact_predictions_inline_notebook",
        "predictions": predictions,
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(output, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"Wrote {len(predictions)} predictions to {output_path}")
    return output

## 11. Execute End-to-End

Chạy cell này để tạo output. Khi smoke test, set `EXACT_LIMIT=5` hoặc sửa `LIMIT = 5` ở cell config. Khi submit, để `LIMIT=None`.

In [30]:
input_path = resolve_input_path()
output_path = pathlib.Path(OUTPUT_PATH)
print("Input:", input_path)
print("Output:", output_path)

prediction_output = run_predictions(
    input_path=input_path,
    output_path=output_path,
    limit=LIMIT,
)

prediction_output["predictions"][:2]

Input: ../datasets/exact/Logic_Based_Educational_Queries_inference.json
Output: ../../../outputs/logic/predictions.json
Loading local tokenizer: Qwen/Qwen2.5-1.5B-Instruct
Loading local model: Qwen/Qwen2.5-1.5B-Instruct with {'torch_dtype': torch.float32, 'device_map': 'auto'}


Loading weights: 100%|██████████| 338/338 [00:06<00:00, 53.84it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


Loaded local model in 7.7s
Running 808 instances | llm_enabled=True | require_llm=True
Local generation finished: tokens=7 elapsed=61.4s


RuntimeError: LLM premise translation failed and fallback is disabled: LLM output did not contain a complete JSON object: ```json
{
    "pred

## 13. Current Notebook Assessment

Notebook hiện tại đã làm được:

1. **Chạy end-to-end trong một notebook Kaggle**
   - Không cần import package `exact` từ repo.
   - Có schema, router, parser, LLM autoformalizer, KB, solver, explanation và batch runner inline.

2. **LLM-first autoformalization**
   - Ưu tiên LLM local/OpenAI-compatible để dịch NL premises/question/options sang Horn IR.
   - Có normalizer để sửa các lỗi JSON phổ biến của local model: lệch key, thiếu wrapper, `predicate` thay vì `pred`, `claim` thay vì `goal`, v.v.

3. **Premise-level LLM cache**
   - Premises được dịch một lần theo `premise_hash`.
   - Các question dùng chung premise group sẽ tái sử dụng cached `KnowledgeBase`.
   - Query/options được dịch bằng prompt ngắn hơn dựa trên predicate inventory của KB.
   - Đây là tối ưu quan trọng để giảm latency trên Kaggle T4x2.

4. **Symbolic reasoning có unification**
   - Forward chaining solver khớp được rule có biến như `completed_core_curriculum(?x)` với fact `completed_core_curriculum(sophia)`.
   - Có proof trace, supporting premises, `fol`, `cot`, `premises`.

5. **MCQ path đúng format**
   - Detect A-D options.
   - Translate/evaluate từng option.
   - Trả `A/B/C/D`, không trả `Yes/No/Unknown` cho MCQ.

6. **Fallback an toàn**
   - Nếu `REQUIRE_LLM=False`, LLM lỗi thì fallback heuristic parser.
   - Nếu `REQUIRE_LLM=True`, lỗi LLM sẽ fail sớm để tránh sinh output giả.

Những điểm còn thiếu để cạnh tranh Top 10:

1. **Premise translation quality vẫn là bottleneck chính**
   - Cần đo trên training set: LLM formalization có tạo đúng predicate/rule không.
   - Nên tạo dev split 80–100 records và đo answer EM + premise support proxy.

2. **Chưa có verifier/repair loop**
   - Hiện chỉ normalize JSON shape.
   - Chưa check predicate arity, free variables, predicate drift, goal predicate có tồn tại trong KB.
   - Chưa có repair prompt khi formalization sai.

3. **Chưa có k-sampling/voting**
   - LINC cho thấy sample nhiều formalization rồi vote bằng solver output thường ổn hơn single-shot.
   - Nhưng cần cân bằng với 60s/request.

4. **Chưa xử lý đầy đủ conditional MCQ options**
   - Option dạng `If A then B` hiện vẫn bị ép thành goal atom hoặc phụ thuộc LLM canonicalization.
   - Muốn chuẩn hơn cần conditional-goal proving: tạm thêm antecedent vào KB rồi prove consequent.

5. **Chưa có Tier-2 FOL prover**
   - Forward chainer chỉ mạnh với Horn rules.
   - Các case `OR`, existential, nested negation, contraposition/general FOL cần Prover9/Mace4 hoặc Z3 finite-domain encoding.

6. **Latency vẫn cần benchmark thật**
   - Premise cache giảm số lần dịch premise, nhưng Qwen 7B trên T4x2 vẫn có thể chậm.
   - Cần log riêng: premise translation time, query translation time, solve time, cache hit rate.
   - Nếu query stage vẫn >60s, cần giảm `LLM_MAX_TOKENS`, prompt ngắn hơn, hoặc dùng Qwen 1.5B cho full-run baseline.

7. **Submission robustness**
   - Cần đảm bảo output official-only đúng format organizer yêu cầu.
   - Cần chạy full 808 với `LIMIT=None` và kiểm tra không instance nào crash.

Khuyến nghị bước tiếp theo:

1. Chạy smoke `LIMIT=5` với cache override để xem query generation còn mất bao lâu.
2. Nếu ổn, chạy `LIMIT=50` để đo cache hit rate và answer distribution.
3. Thêm validator deterministic cho LLM output trước khi solver chạy.
4. Sau validator mới tính tới repair hoặc k-sampling.

## 12. Optional Submission Export

Nếu organizer cần file chỉ gồm official fields, chạy cell này để tạo thêm `exact_official_predictions.json` trong Kaggle working directory.

In [ ]:
official_output_path = pathlib.Path(str(output_path).replace(".json", "_official.json"))
official_payload = {
    "source": prediction_output["source"],
    "count": prediction_output["count"],
    "predictions": [item["official"] for item in prediction_output["predictions"]],
}
official_output_path.write_text(
    json.dumps(official_payload, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("Official-only output:", official_output_path)
official_payload["predictions"][:2]